<a href="https://colab.research.google.com/github/mizuba1r/Accesstack-Report-Generator/blob/main/Accessibility_Report_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# DEPENDENCIES
# =========================================================
!apt-get update -qq

!apt-get install -y -qq \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpangocairo-1.0-0 \
    libpango-1.0-0 \
    libcairo2 \
    libatspi2.0-0 \
    libgtk-3-0 \
    libnss3 \
    libxss1 \
    libxtst6

!pip install -q PyDrive2 playwright nest_asyncio
!playwright install chromium

# =========================================================
# IMPORTS
# =========================================================

import json
import csv
import os
import re
import asyncio
import nest_asyncio
from datetime import datetime

from google.colab import auth
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials
from urllib.parse import urlparse

from playwright.async_api import async_playwright

# =========================================================
# ENABLE ASYNCIO IN COLAB
# =========================================================

nest_asyncio.apply()

# =========================================================
# GOOGLE AUTHENTICATION
# =========================================================

auth.authenticate_user()

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()

drive = GoogleDrive(gauth)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# =========================================================
# CONFIGURATION
# =========================================================

INPUT_FILE = "output.json"

OUTPUT_FILE = "import.csv"

LOCAL_SCREENSHOT_FOLDER = "/content/screenshots"

os.makedirs(LOCAL_SCREENSHOT_FOLDER, exist_ok=True)

# =========================================================
# LOAD JSON
# =========================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

scan_url = data.get("url", "")

In [ ]:
# =========================================================
# CREATE GOOGLE DRIVE FOLDER
# =========================================================
parsed_url = urlparse(scan_url)

website_name = parsed_url.netloc

# Remove common prefixes
website_name = website_name.replace("www.", "")

# Replace invalid Google Drive filename chars
website_name = re.sub(
    r'[^a-zA-Z0-9._-]',
    '_',
    website_name
)

date_str = datetime.now().strftime("%Y-%m-%d")

folder_metadata = {
    "title": f"Audit Screenshots: {website_name} " f"{date_str}",
    "mimeType": "application/vnd.google-apps.folder"
}

drive_folder = drive.CreateFile(folder_metadata)
drive_folder.Upload()

FOLDER_ID = drive_folder["id"]

print(f"Google Drive Folder ID: {FOLDER_ID}")

# =========================================================
# UPLOAD FILE TO GOOGLE DRIVE
# =========================================================

def upload_to_drive(local_path, filename):

    drive_file = drive.CreateFile({
        "title": filename,
        "parents": [{"id": FOLDER_ID}]
    })

    drive_file.SetContentFile(local_path)

    drive_file.Upload()

    # -----------------------------------------------------
    # SHARE SETTINGS:
    # Anyone with link can view
    # -----------------------------------------------------

    drive_file.InsertPermission({
        "type": "anyone",
        "value": "anyone",
        "role": "reader",
        "withLink": True
    })

    # Return shareable link
    return drive_file["alternateLink"]

Google Drive Folder ID: 1WSw4Vr1iGCguAMjhL51V3SwRpDZ3FPzz


In [ ]:
# =========================================================
# HELPER
# =========================================================

def safe_filename(text):
    return re.sub(r'[^a-zA-Z0-9_-]', '_', text)

# =========================================================
# ASYNC PLAYWRIGHT PROCESSING
# =========================================================

async def process_accessibility_issues():

    rows = []

    issue_counter = 1

    async with async_playwright() as playwright:

        browser = await playwright.chromium.launch(
            headless=True
        )

        page = await browser.new_page(
            viewport={"width": 1440, "height": 3000}
        )

        print(f"Opening: {scan_url}")

        await page.goto(
            scan_url,
            wait_until="networkidle",
            timeout=120000
        )

        # -------------------------------------------------
        # PROCESS WCAG FAILURES
        # -------------------------------------------------

        for result in data.get("results", []):

            if result.get("status") != "fail":
                continue

            wcag_number = result.get("wcagNumber", "")
            wcag_name = result.get("wcagName", "")

            for requirement in result.get(
                "requirementsFailed",
                []
            ):

                requirement_key = requirement.get(
                    "requirementKey",
                    ""
                )

                requirement_description = requirement.get(
                    "requirementDescription",
                    ""
                )

                instances = requirement.get(
                    "instances",
                    []
                )

                # -----------------------------------------
                # CREATE ISSUE WITHOUT INSTANCES
                # -----------------------------------------

                if not instances:

                    rows.append({
                        "Issue No.": issue_counter,
                        "URL": scan_url,
                        "Environment": "",
                        "Summary": (
                            f"{wcag_number} - "
                            f"{requirement_description}"
                        ),
                        "Priority": "Medium",
                        "Location": "",
                        "Affected Code": "",
                        "Affected Users": "",
                        "Description": (
                            f"WCAG Criterion: "
                            f"{wcag_number} - {wcag_name}\n\n"
                            f"Requirement Key: "
                            f"{requirement_key}\n"
                            f"Requirement Description: "
                            f"{requirement_description}"
                        ),
                        "Recommendation": "",
                        "Screenshot": ""
                    })

                    print(
                        f"Created issue {issue_counter} "
                        f"(no instances)"
                    )

                    issue_counter += 1

                # -----------------------------------------
                # PROCESS EACH INSTANCE
                # -----------------------------------------

                for instance in instances:

                    path = instance.get("path", "")
                    snippet = instance.get("snippet", "")
                    comment = instance.get("comment", "")

                    screenshot_link = ""

                    # -------------------------------------
                    # TAKE SCREENSHOT
                    # -------------------------------------

                    try:

                      locator = page.locator(path).first

                      await locator.scroll_into_view_if_needed()

                      # Get element bounding box
                      box = await locator.bounding_box()

                      if box is None:
                          raise Exception("Could not determine element position")

                      # ---------------------------------
                      # ADD PADDING AROUND ELEMENT
                      # ---------------------------------

                      padding = 25

                      page_width = page.viewport_size["width"]
                      page_height = page.viewport_size["height"]

                      clip_x = max(box["x"] - padding, 0)
                      clip_y = max(box["y"] - padding, 0)

                      clip_width = min(
                          box["width"] + (padding * 2),
                          page_width - clip_x
                      )

                      clip_height = min(
                          box["height"] + (padding * 2),
                          page_height - clip_y
                      )

                      screenshot_name = (
                          f"issue_{issue_counter}_"
                          f"{safe_filename(wcag_number)}.png"
                      )

                      screenshot_path = os.path.join(
                          LOCAL_SCREENSHOT_FOLDER,
                          screenshot_name
                      )

                      # ---------------------------------
                      # TAKE CLIPPED PAGE SCREENSHOT
                      # ---------------------------------

                      await page.screenshot(
                          path=screenshot_path,
                          clip={
                              "x": clip_x,
                              "y": clip_y,
                              "width": clip_width,
                              "height": clip_height
                          }
                      )
                      # ---------------------------------
                      # UPLOAD TO GOOGLE DRIVE
                      # ---------------------------------

                      screenshot_link = upload_to_drive(
                          screenshot_path,
                          screenshot_name
                      )

                      print(
                          f"Uploaded screenshot "
                          f"for issue {issue_counter}"
                      )

                    except Exception as e:

                        print(
                            f"Could not capture screenshot "
                            f"for issue {issue_counter}: {e}"
                        )

                    # -------------------------------------
                    # DESCRIPTION
                    # -------------------------------------

                    description = f"""
WCAG Criterion: {wcag_number} - {wcag_name}

Requirement Key:
{requirement_key}

Requirement Description:
{requirement_description}

Affected Element Path:
{path}
""".strip()

                    if comment:

                        description += (
                            f"\n\nAdditional Notes:\n{comment}"
                        )

                    # -------------------------------------
                    # ADD CSV ROW
                    # -------------------------------------

                    rows.append({
                        "Issue No.": issue_counter,
                        "URL": scan_url,
                        "Environment": "",
                        "Summary": (
                            f"{wcag_number} - "
                            f"{requirement_description}"
                        ),
                        "Priority": "Medium",
                        "Location": "",
                        "Affected Code": snippet,
                        "Affected Users": "",
                        "Description": description,
                        "Recommendation": "",
                        "Screenshot": screenshot_link
                    })

                    print(
                        f"Created CSV row for issue "
                        f"{issue_counter}"
                    )

                    issue_counter += 1

        await browser.close()

    return rows





In [ ]:
# =========================================================
# RUN PROCESS
# =========================================================

rows = asyncio.get_event_loop().run_until_complete(
    process_accessibility_issues()
)

# =========================================================
# CSV EXPORT
# =========================================================

fieldnames = [
    "Issue No.",
    "URL",
    "Environment",
    "Summary",
    "Priority",
    "Location",
    "Affected Code",
    "Affected Users",
    "Description",
    "Recommendation",
    "Screenshot"
]

with open(
    OUTPUT_FILE,
    "w",
    newline="",
    encoding="utf-8"
) as csvfile:

    writer = csv.DictWriter(
        csvfile,
        fieldnames=fieldnames,
        quoting=csv.QUOTE_ALL
    )

    writer.writeheader()

    writer.writerows(rows)

# =========================================================
# COMPLETE
# =========================================================

print("\n====================================")
print(f"Created Jira CSV file: {OUTPUT_FILE}")
print(f"Total issues created: {len(rows)}")
print("Done.")
print("====================================")

Opening: https://emcal.ng/
Uploaded screenshot for issue 1
Created CSV row for issue 1
Uploaded screenshot for issue 2
Created CSV row for issue 2
Uploaded screenshot for issue 3
Created CSV row for issue 3
Uploaded screenshot for issue 4
Created CSV row for issue 4
Uploaded screenshot for issue 5
Created CSV row for issue 5
Uploaded screenshot for issue 6
Created CSV row for issue 6
Could not capture screenshot for issue 7: Locator.scroll_into_view_if_needed: Unexpected token "" while parsing css selector "". Did you mean to CSS.escape it?
Call log:
  - waiting for  >> nth=0

Created CSV row for issue 7
Uploaded screenshot for issue 8
Created CSV row for issue 8
Uploaded screenshot for issue 9
Created CSV row for issue 9
Uploaded screenshot for issue 10
Created CSV row for issue 10

Created Jira CSV file: import.csv
Total issues created: 10
Done.
